# 🏋️ Train my from-scratch AI — Kaggle background run

Designed for **Save & Run All (Commit)**: press it once, close the app,
turn off your phone. Kaggle runs everything on its servers (free GPU) and
keeps the results as notebook output.

**Pipeline (fully automatic, no interaction):**
1. Get the code from GitHub
2. Download ~70 public-domain books + real Python source code + chat data
3. Train our own BPE tokenizer (vocab 8192) and pack tokens
4. Stage A: pretrain a ~20M-param Transformer from RANDOM weights (time-boxed)
5. Stage B: chat fine-tune (assistant behavior, code Q&A, honest IDK)
6. Export: release checkpoint + pure-NumPy phone version

**Resume across runs:** if a previous run's output is attached as an input
dataset, training auto-resumes from its checkpoint.

Everything is from scratch — no pretrained models, no AI APIs.
Output files land in the notebook's **Output** tab when the run finishes.

In [ ]:
# ===== 1. Setup: code + GPU check =====
import os, sys, glob, time, shutil, subprocess

T0 = time.time()
TIME_BUDGET_H = 10.5          # leave margin inside Kaggle's 12h limit
REPO = "https://github.com/debzitsu-ship-it/Project-lmarena.git"
BRANCH = "arena/01a0011c-project-lmarena"

os.chdir("/kaggle/working")
if not os.path.exists("Project-lmarena"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO], check=True)
os.chdir("Project-lmarena")
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, os.getcwd())

import torch
DEVICE_OK = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if DEVICE_OK else "NONE (enable GPU in notebook settings!)")

# ===== restore checkpoints from a previous run, if attached as input =====
os.makedirs("my_ai/checkpoints", exist_ok=True)
restored = []
for src in glob.glob("/kaggle/input/**/latest.pt", recursive=True) + \
           glob.glob("/kaggle/input/**/pretrain_latest.pt", recursive=True):
    shutil.copy(src, "my_ai/checkpoints/latest.pt"); restored.append(src); break
for src in glob.glob("/kaggle/input/**/tokenizer.json", recursive=True):
    os.makedirs("my_ai/data/processed", exist_ok=True)
    shutil.copy(src, "my_ai/data/processed/tokenizer.json"); restored.append(src); break
print("restored from previous run:" if restored else "fresh start —", restored or "no previous output attached")

In [ ]:
# ===== 2. Data: books + python code + chat templates =====
import urllib.request, subprocess, os
subprocess.run([sys.executable, "-m", "my_ai.data.fetch_gitenberg"], check=False)
subprocess.run([sys.executable, "-m", "my_ai.data.fetch_python_code"], check=False)

# Kaggle has open internet: add extra Gutenberg books directly
EXTRA_IDS = [1080, 2542, 5200, 16389, 902, 408, 1232, 844, 120, 2591]
for bid in EXTRA_IDS:
    dest = f"my_ai/data/raw/gutenberg_{bid}.txt"
    if os.path.exists(dest): continue
    for url in (f"https://www.gutenberg.org/cache/epub/{bid}/pg{bid}.txt",
                f"https://www.gutenberg.org/files/{bid}/{bid}-0.txt"):
        try:
            txt = urllib.request.urlopen(url, timeout=30).read().decode("utf-8", "replace")
            s = txt.find("*** START"); e = txt.find("*** END")
            if s != -1: txt = txt[txt.find("\n", s):]
            if e != -1: txt = txt[:e]
            open(dest, "w").write(txt)
            break
        except Exception:
            continue
subprocess.run([sys.executable, "-m", "my_ai.data.make_chat_data"], check=True)
total = sum(os.path.getsize(os.path.join("my_ai/data/raw", f)) for f in os.listdir("my_ai/data/raw"))
print(f"corpus: {total/1e6:.1f} MB, {len(os.listdir('my_ai/data/raw'))} files")

In [ ]:
# ===== 3. Tokenizer + packing (skipped if restored from previous run) =====
import os, subprocess, sys
if os.path.exists("my_ai/data/processed/tokenizer.json") and os.path.exists("my_ai/checkpoints/latest.pt"):
    # resuming: rebuild token files with the SAME tokenizer (model is married to it)
    print("resuming with previous tokenizer — repacking tokens...")
    from my_ai.tokenizer.tokenizer import load_tokenizer
    from my_ai.data.dataset import load_corpus, split_corpus, tokenize_and_pack
    import json
    tok = load_tokenizer("my_ai/data/processed/tokenizer.json")
    docs = load_corpus(["my_ai/data/raw"])
    tr, va, te = split_corpus(docs)
    meta = {"vocab_size": tok.vocab_size}
    for name, subset in (("train", tr), ("val", va), ("test", te)):
        arr = tokenize_and_pack(subset, tok, f"my_ai/data/processed/{name}.bin")
        meta[f"{name}_tokens"] = int(len(arr))
    json.dump(meta, open("my_ai/data/processed/meta.json", "w"))
    print(meta)
else:
    subprocess.run([sys.executable, "-m", "my_ai.prepare_data",
                    "--input", "my_ai/data/raw", "--out", "my_ai/data/processed",
                    "--vocab-size", "8192", "--tokenizer-sample-chars", "2000000"], check=True)
subprocess.run([sys.executable, "-m", "my_ai.prepare_finetune",
                "--data", "my_ai/data/processed", "--raw", "my_ai/data/raw",
                "--chat-frac", "0.7"], check=True)

In [ ]:
# ===== 4. STAGE A: pretrain ~20M params, time-boxed chunks =====
import os, time, subprocess, sys
TOTAL_STEPS, CHUNK = 20000, 2000
done_this_run = 0
while done_this_run < TOTAL_STEPS:
    elapsed_h = (time.time() - T0) / 3600
    if elapsed_h > TIME_BUDGET_H - 1.0:   # keep 1h for finetune+export
        print(f"time budget reached ({elapsed_h:.1f}h) — moving on to Stage B")
        break
    resume = os.path.exists("my_ai/checkpoints/latest.pt")
    cmd = [sys.executable, "-m", "my_ai.train",
           "--config", "my_ai/configs/small_20m.json",
           "--data", "my_ai/data/processed",
           "--steps", str(CHUNK), "--batch-size", "32", "--lr", "6e-4",
           "--sample-prompt", "def add(a, b):"]
    if resume:
        cmd += ["--resume", "my_ai/checkpoints/latest.pt"]
    rc = subprocess.run(cmd).returncode
    if rc != 0:
        print("training chunk failed, rc", rc); break
    done_this_run += CHUNK
    print(f"=== {done_this_run} steps done this session ===")
print("Stage A session complete")

In [ ]:
# ===== 5. STAGE B: chat fine-tune =====
import subprocess, sys, os
if os.path.exists("my_ai/checkpoints/latest.pt"):
    subprocess.run([sys.executable, "-m", "my_ai.finetune",
                    "--base", "my_ai/checkpoints/latest.pt",
                    "--data", "my_ai/data/processed",
                    "--steps", "2000", "--lr", "1e-4", "--batch-size", "32"], check=False)
else:
    print("no pretrained checkpoint — Stage B skipped")

In [ ]:
# ===== 6. Quality check + export everything to /kaggle/working =====
import os, shutil, subprocess, sys, torch
OUT = "/kaggle/working/model_output"
os.makedirs(OUT, exist_ok=True)

best_chat = "my_ai/checkpoints/chat/best.pt"
src = best_chat if os.path.exists(best_chat) else "my_ai/checkpoints/latest.pt"
if os.path.exists(src):
    ckpt = torch.load(src, map_location="cpu", weights_only=False)
    lean = {"model_state": ckpt["model_state"], "model_config": ckpt["model_config"],
            "step": ckpt.get("step"), "note": "Kaggle 20M two-stage run, from scratch"}
    torch.save(lean, f"{OUT}/model_release.pt")
    # phone (pure NumPy) export
    subprocess.run([sys.executable, "-m", "my_ai.inference.export_numpy",
                    "--checkpoint", f"{OUT}/model_release.pt",
                    "--out", f"{OUT}/model_numpy.npz"], check=False)
    tok_src = ("my_ai/checkpoints/chat/tokenizer.json"
               if os.path.exists("my_ai/checkpoints/chat/tokenizer.json")
               else "my_ai/data/processed/tokenizer.json")
    shutil.copy(tok_src, f"{OUT}/tokenizer.json")
    # raw checkpoints for resuming next session
    shutil.copy("my_ai/checkpoints/latest.pt", f"{OUT}/latest.pt")

    # quick sample generations into the log
    from my_ai.training.trainer import load_checkpoint
    from my_ai.tokenizer.tokenizer import load_tokenizer
    from my_ai.inference.generate import generate_text
    model, _ = load_checkpoint(f"{OUT}/model_release.pt")
    tok = load_tokenizer(f"{OUT}/tokenizer.json")
    model.eval()
    for p in ["<|user|>Tell me about yourself.<eos><|assistant|>",
              "<|user|>Write a Python function that adds two numbers.<eos><|assistant|>",
              "<|user|>What is quantum physics?<eos><|assistant|>",
              "Once upon a time"]:
        torch.manual_seed(0)
        print("PROMPT:", p)
        print("OUTPUT:", generate_text(model, tok, p, max_new_tokens=60,
                                       temperature=0.7, top_k=40)[len(p):].strip()[:300])
        print("---")
    print("\nAll outputs in", OUT, ":", os.listdir(OUT))
else:
    print("nothing to export")